### Config — every parameter of the pipeline lives here
Used by all notebooks via `%run ../../config/config`. Change a value here and re-run; no code edits needed.

In [0]:
# ---------- catalog / storage
CATALOG = "nyctaxi_zsi"
VOLUME_PATH = f"/Volumes/{CATALOG}/00_landing/nyc_taxi_stress"
RAW_DIR = f"{VOLUME_PATH}/raw"
OUTPUT_DIR = f"{VOLUME_PATH}/outputs"

TRIPS_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-03.parquet"
ZONES_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
TRIPS_FILE = f"{RAW_DIR}/yellow_tripdata_2025-03.parquet"
ZONES_FILE = f"{RAW_DIR}/taxi_zone_lookup.csv"

# ---------- time
TIMEZONE = "America/New_York"      # source timestamps are NY wall-clock with no zone attached
OP_DATE_START = "2025-03-01"       # operational dates in scope
OP_DATE_END = "2025-03-31"

# ---------- silver: data-quality thresholds
PERIOD_START = "2025-03-01 00:00:00"
PERIOD_END = "2025-04-01 04:30:00"    # keeps operational day 2025-03-31 whole
MAX_DURATION_MIN = 360

# ---------- lookup rows that are not physical taxi zones
NON_PHYSICAL_BOROUGHS = ["Unknown", "N/A", ""]
NON_PHYSICAL_ZONES = ["N/A", "Outside of NYC", "NV", ""]

# ---------- gold: Zone Stress Index settings ("spec" = assignment, "conventional" = Q4)
ZSI_SETTINGS = {
    "spec": {
        "op_day_start": "04:30",
        "window_minutes": 20,
        "am_window_start_min": "06:50",     # inclusive
        "am_window_start_max": "09:50",     # inclusive
        "am_weekdays": [1, 2, 3, 4, 5],     # Mon..Fri of the operational date
        "quantile": 0.9,
        "p_offset": 1,                      # sqrt(P + p_offset)
        "smoothing_constant": 3,            # sqrt(4D + smoothing_constant)
        "expected_am_windows": 210,         # set to None if you change the grid
        "table_suffix": "",
    },
    "conventional": {
        "op_day_start": "00:00",
        "window_minutes": 20,
        "am_window_start_min": "06:40",
        "am_window_start_max": "09:40",
        "am_weekdays": [1, 2, 3, 4, 5],
        "quantile": 0.9,
        "p_offset": 1,
        "smoothing_constant": 3,
        "expected_am_windows": 210,
        "table_suffix": "_conventional",
    },
}


def tbl(schema: str, name: str) -> str:
    """Full table name, e.g. tbl('02_silver', 'fact_trips') -> nyctaxi_zsi.02_silver.fact_trips"""
    return f"{CATALOG}.{schema}.{name}"


# Session zone UTC => Spark timestamp arithmetic is plain wall-clock arithmetic (UTC has no DST).
# Real instants are always explicit: to_utc_timestamp(ts, 'America/New_York').
spark.conf.set("spark.sql.session.timeZone", "UTC")
print(f"config loaded: catalog={CATALOG}, volume={VOLUME_PATH}")